In [2]:
using PowerModels

sys = PowerModels.parse_file("case2383wp-Islanded.m")

[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 2243 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 2243 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 1881 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 1881 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 1907 from -360.0 to -60.0 deg.
[warn | PowerModels]: this code only supports angmax values in -90 deg. to 90 deg., tightening the value on branch 1907 from 360.0 to 60.0 deg.
[warn | PowerModels]: this code only supports angmin values in -90 deg. to 90 deg., tightening the value on branch 599 from -360.0

Excessive output truncated after 524434 bytes.

Dict{String, Any} with 13 entries:
  "bus"            => Dict{String, Any}("1766"=>Dict{String, Any}("zone"=>4, "b…
  "source_type"    => "matpower"
  "name"           => "case2383wp"
  "dcline"         => Dict{String, Any}()
  "source_version" => "2"
  "gen"            => Dict{String, Any}("306"=>Dict{String, Any}("ncost"=>0, "q…
  "branch"         => Dict{String, Any}("2243"=>Dict{String, Any}("br_r"=>0.011…
  "storage"        => Dict{String, Any}()
  "switch"         => Dict{String, Any}()
  "baseMVA"        => 100
  "per_unit"       => true
  "shunt"          => Dict{String, Any}()
  "load"           => Dict{String, Any}("1766"=>Dict{String, Any}("source_id"=>…

In [21]:
# Disconnect Branches Between Zones to create islands

using PowerModels


# sys = PowerModels.parse_file("case2383wp.m")

branch_keys = collect(keys(sys["branch"]))

# # Loop through all branches
for branch_id in branch_keys
    # Extract current branch
    branch = sys["branch"][branch_id]

    # Departure bus
    f = string(branch["f_bus"])

    # Destination bus
    t = string(branch["t_bus"])

    # Departure zone (1-6)
    f_zone = sys["bus"][f]["zone"]

    # Destination zone (1-6)
    t_zone = sys["bus"][t]["zone"]

    # If zones differ, we need to check if zone 6, if not, delete the branch
    if f_zone != t_zone
        
        # Case 1: From bus is zone 6 → adopt zone from t_bus
        if f_zone == 6
            sys["bus"][f]["zone"] = t_zone
            continue  # now zones match, so don't delete

        # Case 2: To bus is zone 6 → adopt zone from f_bus
        elseif t_zone == 6
            sys["bus"][t]["zone"] = f_zone
            continue
        end

        # Otherwise: both are non-6 and different → delete branch

        println("Disconnecting branch:", branch_id, "  from:", string(branch["f_bus"]), "  to:", string(branch["t_bus"]), "   f zone:", sys["bus"][f]["zone"], "   t_zone:", sys["bus"][t]["zone"])
            
        delete!(sys["branch"], branch_id)
    end
end


println("After filtering, remaining branches: ", length(sys["branch"]))

Disconnecting branch:39  from:111  to:11   f zone:4   t_zone:1
Disconnecting branch:1019  from:670  to:1469   f zone:2   t_zone:3
Disconnecting branch:1085  from:1430  to:738   f zone:3   t_zone:2
Disconnecting branch:61  from:18  to:101   f zone:1   t_zone:3
Disconnecting branch:116  from:80  to:41   f zone:3   t_zone:2
Disconnecting branch:29  from:29  to:7   f zone:2   t_zone:1
Disconnecting branch:324  from:166  to:139   f zone:5   t_zone:4
Disconnecting branch:620  from:2249  to:315   f zone:5   t_zone:1
Disconnecting branch:1065  from:715  to:1190   f zone:2   t_zone:3
Disconnecting branch:325  from:174  to:139   f zone:5   t_zone:4
Disconnecting branch:86  from:27  to:74   f zone:2   t_zone:3
Disconnecting branch:1041  from:689  to:1402   f zone:2   t_zone:3
Disconnecting branch:960  from:625  to:1348   f zone:2   t_zone:3
Disconnecting branch:60  from:18  to:76   f zone:1   t_zone:3
Disconnecting branch:933  from:606  to:926   f zone:2   t_zone:3
Disconnecting branch:659  from:

In [10]:
# Extract islands with calc_connected_components() and test samples to verify zones
using PowerModelsDistribution

# Find the islands
components = PowerModels.calc_connected_components(sys)
println("Found $(length(components)) connected components:")

# println("Components: $(components) looks like: $(island_list[5])")


# Collect components to allow for indexing list of islands
island_list = collect(components)

# Pick a sample island and collect indices of that island to allow for indexing buses
sample_island = collect(island_list[5])

# Pick a sample bus and collect indices of that bus to allow for indexing bus attributes 
sample_bus = collect(sample_island[1])

# Select the bus_id of our sample bus and convert it's id to a string
sample_bus_id = string(sample_bus[1])


println("Sample bus $(sample_bus_id) is in Zone: $(sys["bus"][sample_bus_id]["zone"])")

println("The Zone of sample bus bus $(sample_bus_id) looks like: $(island_list[sys["bus"][sample_bus_id]["zone"]])")


# A. Keep only the buses in your target set (Instant operation)
filter!(p -> parse(Int, p.first) in target_island_ids, island_sys["bus"])



Found 5 connected components:
Sample bus 2261 is in Zone: 5
The Zone of sample bus bus 2261 looks like: Set([2261, 2288, 2158, 2350, 2312, 2324, 2236, 2297, 2164, 2291, 2300, 2377, 2271, 2123, 2325, 2174, 2232, 2285, 2211, 2374, 168, 2132, 2177, 2252, 2356, 2119, 2360, 177, 2249, 2250, 2275, 2154, 2188, 172, 2141, 2382, 2242, 2136, 2381, 2143, 2168, 2336, 2373, 2294, 2166, 2170, 2305, 170, 2213, 2218, 2326, 2153, 313, 2363, 2255, 2341, 2268, 165, 2212, 2372, 2204, 2180, 2199, 2272, 2120, 2156, 2191, 2267, 2316, 2284, 2256, 2259, 2330, 2283, 2354, 2234, 2349, 158, 2220, 2138, 176, 2345, 2161, 2206, 2319, 2311, 2125, 2162, 2304, 2279, 2226, 2303, 2264, 162, 2227, 2321, 2296, 2335, 2140, 2365, 2149, 2124, 157, 2221, 2282, 2228, 2241, 2200, 2151, 2331, 2276, 2175, 2150, 2229, 159, 2195, 2117, 2379, 2351, 2337, 2260, 2222, 2122, 2137, 166, 2370, 2339, 2181, 2258, 2190, 2210, 2209, 2344, 2315, 2318, 2375, 2281, 2182, 2142, 2208, 2239, 2207, 164, 2309, 2317, 2302, 2193, 2238, 2380, 2179, 2173

In [9]:
using Ipopt
# Run Power Flow on the largest island in the grid

# Create a copy of the case
island_sys = deepcopy(sys)

# Identify the largest island
PowerModels.select_largest_component!(island_sys)

# Iterate through all generators and fix ALL infinite bounds
for (k, gen) in island_sys["gen"]
    
    #Fix Lower Bounds (-Inf -> -1e10)
    if gen["qmin"] == -Inf
        gen["qmin"] = -1.0e10
    end
    if gen["pmin"] == -Inf
        gen["pmin"] = -1.0e10
    end

    # Fix Upper Bounds (Inf -> 1e10) 
    if gen["qmax"] == Inf
        gen["qmax"] = 1.0e10
    end
    if gen["pmax"] == Inf
        gen["pmax"] = 1.0e10
    end
end


# Solve Largest Island
result = solve_opf(island_sys, ACPPowerModel, Ipopt.Optimizer)

[info | PowerModels]: deactivating bus 1299 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 1370 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 1085 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 1466 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 844 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 1636 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 1896 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 2063 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 937 due to dangling bus without generation, load or storage
[info | PowerModels]: deactivating bus 1341 due to dangling bus without generation, load or storage
[i

Dict{String, Any} with 8 entries:
  "solve_time"         => 16.357
  "optimizer"          => "Ipopt"
  "termination_status" => LOCALLY_SOLVED
  "dual_status"        => FEASIBLE_POINT
  "primal_status"      => FEASIBLE_POINT
  "objective"          => 1.86815e6
  "solution"           => Dict{String, Any}("baseMVA"=>100, "branch"=>Dict{Stri…
  "objective_lb"       => -Inf

In [26]:

island_sys_test = deepcopy(sys)


"""
determines the largest connected component of the network and turns everything else off
"""
function select_component_direct!(data::Dict{String, <:Any}, index::Int)
    
    # 1. Map out the grid islands directly
    ccs = calc_connected_components(data)

    # 2. Sort from Largest to Smallest
    ccs_order = sort(collect(ccs); by=length, rev=true)

    # 3. Safety check
    if index > length(ccs_order) || index < 1
        error("Index $index out of bounds. The network only has $(length(ccs_order)) island(s).")
    end

    selected_cc = ccs_order[index]
    println("Isolating island $index (Size: $(length(selected_cc)) buses)...")

    # 4. Directly modify the bus dictionary
    for (i, bus) in data["bus"]
        if !(bus["index"] in selected_cc)
            bus["bus_type"] = 4 # Deactivate bus
        end
    end

    # 5. Run the native cleanup tools directly on the data dict
    propagate_topology_status!(data)
    correct_reference_buses!(data)

    return data
end
PowerModels.silence()
select_component_direct!(island_sys_test, 2)

print(island_sys)

# result = solve_opf(island_sys, ACPPowerModel, Ipopt.Optimizer)

Isolating island 2 (Size: 570 buses)...
Dict{String, Any}("bus" => Dict{String, Any}("306" => Dict{String, Any}("zone" => 1, "bus_i" => 306, "bus_type" => 1, "vmax" => 1.12, "source_id" => Any["bus", 306], "area" => 1, "vmin" => 0.95, "index" => 306, "va" => -0.3995814036315284, "vm" => 1.1183727, "base_kv" => 110.0), "1886" => Dict{String, Any}("zone" => 4, "bus_i" => 1886, "bus_type" => 1, "vmax" => 1.12, "source_id" => Any["bus", 1886], "area" => 1, "vmin" => 0.95, "index" => 1886, "va" => -0.5090825231436116, "vm" => 1.1180174, "base_kv" => 110.0), "1" => Dict{String, Any}("zone" => 1, "bus_i" => 1, "bus_type" => 1, "vmax" => 1.11, "source_id" => Any["bus", 1], "area" => 1, "vmin" => 0.95, "index" => 1, "va" => -0.026088617917462843, "vm" => 1.0945877, "base_kv" => 220.0), "519" => Dict{String, Any}("zone" => 1, "bus_i" => 519, "bus_type" => 1, "vmax" => 1.12, "source_id" => Any["bus", 519], "area" => 1, "vmin" => 0.95, "index" => 519, "va" => -0.24139509257993413, "vm" => 1.116571

Excessive output truncated after 524288 bytes.

Dict{String, Any}("zone" => 3, "bus_i" => 1161, "bus_type" => 1, "vmax" => 1.12, "source_id" => Any["bus", 1161], "area" => 1, "vmin" => 0.95, "index" => 1161, "va" => -0.0735111719535695, "vm" => 1.093702, "base_kv" => 110.0), "1096" => Dict{String, Any}("zone" => 3, "bus_i" => 1096, "bus_type" => 1, "vmax" => 1.12, "source_id" => Any["bus", 1096], "area" => 1, "vmin" => 0.95, "index" => 1096, "va" => -0.059038973354531864, "vm" => 1.0965851, "base_kv" => 110.0), "1173" => Dict{String, Any}("zone" => 3, "bus_i" => 1173, "bus_type" => 1, "vmax" => 1.12, "source_id" => Any["bus", 1173], "area" => 1, "vmin" => 0.95, "index" => 1173, "va" => -0.08972048977580012, "vm" => 1.0904726, "base_kv" => 110.0), "1182" => Dict{String, Any}("zone" => 3, "bus_i" => 1182, "bus_type" => 2, "vmax" => 1.12, "source_id" => Any["bus", 1182], "area" => 1, "vmin" => 0.95, "index" => 1182, "va" => -0.1098435304220443, "vm" => 1.1011909, "base_kv" => 110.0), "1521" => Dict{String, Any}("zone" => 3, "bus_i" => 